# Action Transition Graph Analysis - Updated with Refined Categories

This notebook analyzes agent action transition patterns using the **refined 10-category system** from `0914_trace_analysis.ipynb`:

1. **Data Exploration (29.7%)** - Exploring patterns, distributions, results analysis
2. **Study Design Analysis (19.7%)** - Understanding experimental setup and conditions
3. **Multi-source Integration (16.8%)** - Combining experimental + computational data
4. **Statistical Analysis (15.8%)** - All statistical tests consolidated
5. **Effect Size Calculation (8.0%)** - Biological significance measures
6. **Debugging (3.9%)** - Data structure validation and troubleshooting
7. **Group Stats Calculation (2.2%)** - Computing group summaries
8. **Results Export (2.0%)** - Saving analysis results
9. **Other Operations (2.0%)** - Miscellaneous operations
10. **Visualization (0.0%)** - Plot creation (rare)

This analysis reveals **how actions flow and transition** in scientific workflows, complementing the frequency analysis with workflow progression insights.

In [ ]:
import re, pickle, warnings, numpy as np, pandas as pd, seaborn as sns, networkx as nx, seaborn as sns, matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Set plotting style
plt.rcParams['figure.dpi'] = 300
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['pdf.use14corefonts'] = True
sns.set_palette("husl")

# Load the successful traces data
print("Loading trace data...")
with open('../successful_traces_mouse.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"Total studies: {len(data)}")

## 1. Integration with Refined Action Categories

This analysis now uses the **exact same categorization system** as `0914_trace_analysis.ipynb` to ensure consistency across analyses. The key improvement is applying the semantically meaningful categories to understand workflow transitions and progressions.

### Refined Categories Applied:
Based on comprehensive code execution block analysis, we use these 10 categories:
1. **Data Exploration** - Exploring patterns, distributions, results analysis
2. **Study Design Analysis** - Understanding experimental setup and conditions  
3. **Multi-source Integration** - Combining experimental + computational data
4. **Statistical Analysis** - All statistical tests consolidated
5. **Effect Size Calculation** - Biological significance measures
6. **Debugging** - Data structure validation and troubleshooting
7. **Group Stats Calculation** - Computing group summaries
8. **Results Export** - Saving analysis results
9. **Other Operations** - Miscellaneous operations
10. **Visualization** - Plot creation

This ensures **perfect consistency** between frequency analysis and workflow transition analysis.

In [ ]:
def categorize_code_block_for_graph(code_content):
    """ Use the same categorization logic from 0914_trace_analysis.ipynb """
    code = code_content.lower()
    lines = code_content.strip().split('\\n')
    
    # Priority-based semantic categorization (most specific first)
    
    # 1. STATISTICAL ANALYSIS (all statistical tests merged)
    if any(stat_test in code for stat_test in [
        'ttest_ind', 'ttest_rel', 'f_oneway', 'mannwhitneyu', 'wilcoxon', 
        'kruskal', 'pearsonr', 'spearmanr', 'ols(', 'sm.ols'
    ]):
        return 'Statistical Analysis'
    
    # 2. EFFECT SIZE CALCULATION  
    if ('cohens_d' in code or 'def cohens_d' in code_content or
        ('effect_size' in code and '=' in code_content)):
        return 'Effect Size Calculation'
    
    # 3. MULTI-SOURCE INTEGRATION (combining different data sources)
    if (('.merge(' in code_content or 'pd.merge(' in code_content) and 
        ('metadata' in code and 'clock' in code)):
        return 'Multi-source Integration'
    
    # 4. GROUP STATS CALCULATION (computing summary statistics by groups)  
    if ('.groupby(' in code_content and 
        any(agg in code_content for agg in ['.describe()', '.mean()', '.size()', '.count()', '.sum()'])):
        return 'Group Stats Calculation'
        
    # 5. VISUALIZATION (creating actual plots)
    if (any(plot_func in code_content for plot_func in [
        'plt.figure(', 'plt.hist(', 'plt.boxplot(', 'plt.scatter(', 
        'plt.plot(', 'sns.boxplot(', 'sns.histplot('
    ]) and ('plt.show(' in code_content or 'plt.savefig(' in code_content)):
        return 'Other Operations'
    
    # 6. RESULTS EXPORT (saving analysis results)
    if ('.to_csv(' in code_content and 
        ('output' in code and 'makedirs' in code_content)):
        return 'Results Export'
    
    # 7-8. DATA INTERPRETATION BREAKDOWN
    # Check for print statements first
    if (code_content.count('print(') >= 2 and
        not any(plot in code_content for plot in ['plt.', 'sns.', 'matplotlib'])):
        
        # 7. STUDY DESIGN ANALYSIS - experimental setup, treatment groups, conditions
        if any(design_term in code for design_term in [
            'treatment groups', 'condition', 'study design', 'sample count',
            'unique', 'genotype', 'experimental design'
        ]):
            return 'Study Design Analysis'
        
        # 8. DATA EXPLORATION - distributions, patterns, group comparisons, results analysis
        elif any(explore_term in code for explore_term in [
            'distribution', 'pattern', 'group comparison', 'samples per',
            'values by', 'analysis', 'comparison', 't_stat', 'p_value', 
            'statistic', 'correlation', 'significant', 'effect', 'summary statistics', 
            'aging', 'clock'
        ]):
            return 'Data Exploration'
    
    # 9. DEBUGGING (checking data structure and troubleshooting)
    if (code_content.count('print(') >= 2 and
        any(debug_term in code for debug_term in [
            'columns', 'shape', 'dtypes', 'head()', 'info()', 'sample', 
            'metadata', 'merged', 'check', 'error', 'missing', 'null'
        ]) and
        not any(plot in code_content for plot in ['plt.', 'sns.', 'matplotlib'])):
        return 'Debugging'
    
    return 'Other Operations'

def extract_action_sequences_with_refined_categories(data, min_sequence_length=5):
    """ Extract action sequences using refined categories from trace analysis """
    
    print(f"Analyzing all {len(data)} studies with refined categorization...")
    
    action_sequences = []
    category_stats = Counter()
    filtered_count = 0
    
    for study_id in tqdm(data.keys(), desc="Extracting refined action sequences"):
        entry = data[study_id]
        if 'result' in entry and len(entry['result']) >= 1:
            messages = entry['result'][0]
            
            sequence = []
            
            for message in messages:
                message_text = str(message)
                
                # Extract code blocks and categorize them
                code_matches = re.findall(r'<execute>(.*?)</execute>', message_text, re.DOTALL)
                for code_block in code_matches:
                    action_category = categorize_code_block_for_graph(code_block)
                    sequence.append(action_category)
                    category_stats[action_category] += 1
            
            # Compress consecutive identical actions to reduce noise
            compressed_sequence = []
            if sequence:
                compressed_sequence = [sequence[0]]
                for i in range(1, len(sequence)):
                    if sequence[i] != sequence[i-1]:
                        compressed_sequence.append(sequence[i])
            
            # Filter out sequences that are too short
            if len(compressed_sequence) >= min_sequence_length:
                action_sequences.append({
                    'study_id': study_id,
                    'original_sequence': sequence,
                    'compressed_sequence': compressed_sequence,
                    'original_length': len(sequence),
                    'compressed_length': len(compressed_sequence),
                    'execution_time': entry['execution_time']
                })
            else:
                filtered_count += 1
    
    print(f"\nFiltered out {filtered_count} sequences shorter than {min_sequence_length} steps")
    return action_sequences, category_stats

# Extract refined action sequences
action_sequences, category_stats = extract_action_sequences_with_refined_categories(data, min_sequence_length=5)

print(f"\n=== REFINED ACTION SEQUENCE ANALYSIS ===\n")
print(f"Extracted {len(action_sequences)} action sequences")
print(f"Average original length: {np.mean([seq['original_length'] for seq in action_sequences]):.1f}")
print(f"Average compressed length: {np.mean([seq['compressed_length'] for seq in action_sequences]):.1f}")
print(f"Compression efficiency: {1 - np.mean([seq['compressed_length']/seq['original_length'] for seq in action_sequences]):.1%}")

print(f"\n=== REFINED ACTION CATEGORIES (from trace analysis) ===\n")
total_actions = sum(category_stats.values())
for category, count in category_stats.most_common():
    percentage = count / total_actions * 100
    print(f"• {category}: {count:,} actions ({percentage:.1f}%)")

# Show example refined workflow
example = action_sequences[0]
print(f"\nExample refined workflow (Study {example['study_id']}):")
print(f"Compressed ({example['compressed_length']} steps): {' → '.join(example['compressed_sequence'][:10])}{'...' if example['compressed_length'] > 10 else ''}")

## 2. Step-by-Step Action Composition Analysis - Updated with Refined Categories

Analyze action composition for each workflow step using the **refined 10-category system** from `0914_trace_analysis.ipynb`. This shows how the workflow progresses from step to step using the semantically meaningful categories.

In [ ]:
def analyze_step_composition_refined(action_sequences, max_steps=30):
    """ Analyze action composition using categories from trace analysis """
    
    # Initialize step composition tracker
    step_composition = {}
    step_counts = {}
    
    for step in range(max_steps):
        step_composition[step] = Counter()
        step_counts[step] = 0
    
    # Track action composition at each step
    for seq_data in action_sequences:
        sequence = seq_data['compressed_sequence']
        
        for step, action in enumerate(sequence[:max_steps]):
            step_composition[step][action] += 1
            step_counts[step] += 1
    
    # Get all refined actions
    all_actions = set()
    for seq_data in action_sequences:
        all_actions.update(seq_data['compressed_sequence'])
    all_actions = sorted(list(all_actions))
    
    # Convert to percentages for visualization
    step_percentages = {}
    for step in range(max_steps):
        step_percentages[step] = {}
        total = step_counts[step]
        if total > 0:
            for action in all_actions:
                count = step_composition[step].get(action, 0)
                step_percentages[step][action] = count / total * 100
        else:
            for action in all_actions:
                step_percentages[step][action] = 0
    
    return step_composition, step_percentages, step_counts, all_actions

# Analyze step composition with refined categories
MAX_STEPS = 30
step_composition, step_percentages, step_counts, all_actions = analyze_step_composition_refined(action_sequences, max_steps=MAX_STEPS)

print("=== STEP-BY-STEP COMPOSITION WITH REFINED CATEGORIES ===")
print(f"Analyzing first {MAX_STEPS} steps of {len(action_sequences)} workflows")

# Show workflow length distribution
workflow_lengths = [len(seq['compressed_sequence']) for seq in action_sequences]
print(f"\nWorkflow length statistics:")
print(f"• Average: {np.mean(workflow_lengths):.1f} steps")
print(f"• Range: {min(workflow_lengths)} - {max(workflow_lengths)} steps")
print(f"• Workflows with ≥{MAX_STEPS} steps: {sum(1 for l in workflow_lengths if l >= MAX_STEPS)} ({sum(1 for l in workflow_lengths if l >= MAX_STEPS)/len(workflow_lengths)*100:.1f}%)")

# Show step participation rates
print(f"\nStep participation rates:")
for step in range(0, MAX_STEPS, 5):
    participation = step_counts[step] / len(action_sequences) * 100
    print(f"• Step {step+1:2d}: {step_counts[step]:4d} workflows ({participation:5.1f}%)")

In [ ]:
# Create step composition visualization with refined categories
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

# Refined color scheme matching the new categories
refined_category_colors = {
    'Data Exploration': '#E74C3C',           # Red - dominant category
    'Study Design Analysis': '#3498DB',      # Blue - second most common
    'Multi-source Integration': '#2ECC71',   # Green - data combining
    'Statistical Analysis': '#9B59B6',       # Purple - statistical work
    'Effect Size Calculation': '#F39C12',    # Orange - effect calculations  
    'Debugging': '#1ABC9C',                  # Teal - debugging activities
    'Group Stats Calculation': '#E67E22',    # Orange-red - group stats
    'Results Export': '#95A5A6',             # Gray - export activities
    'Other Operations': '#34495E',           # Dark gray - miscellaneous
    # 'Visualization': '#16A085'               # Dark teal - visualization
}

# Order actions by frequency from trace analysis for consistent stacking
ordered_actions = [
    'Data Exploration',
    'Study Design Analysis', 
    'Multi-source Integration',
    'Statistical Analysis',
    'Effect Size Calculation',
    'Debugging',
    'Group Stats Calculation',
    'Results Export',
    'Other Operations',
    # 'Visualization'
]

# Build stacked area chart with refined categories
steps = list(range(MAX_STEPS))
bottoms = np.zeros(MAX_STEPS)

for action in ordered_actions:
    if action in all_actions:
        percentages = [step_percentages[step][action] for step in steps]
        color = refined_category_colors.get(action, '#CCCCCC')
        ax.bar(steps, percentages, bottom=bottoms, color=color, alpha=0.8, width=0.8, label=action, edgecolor='white', linewidth=0.3)
        
        bottoms += percentages

# ax.set_xlabel('Workflow Step', fontsize=12)
ax.set_ylabel('Action Composition (%)', fontsize=12)
ax.set_title(f'Step-by-Step Action Composition', fontsize=14, pad=20)

# Improved x-axis labeling
tick_interval = 5
ax.set_xticks(range(tick_interval-1, MAX_STEPS, tick_interval))
ax.set_xticklabels([f'{i+1}' for i in range(tick_interval-1, MAX_STEPS, tick_interval)])

# Enhanced legend
ax.legend(bbox_to_anchor=(1.02, 0.97), loc='upper left', fontsize=10, frameon=True, fancybox=True, shadow=True)

ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(-0.5, MAX_STEPS-0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


plt.tight_layout()
plt.savefig("stepwise action compostion.pdf", dpi=300)
plt.show()

## 3. Fixed Workflow Stage Analysis

Corrected version with proper key handling and enhanced analysis.

In [ ]:
def analyze_fixed_workflow_stages(action_sequences):
    """
    Fixed workflow stage analysis with proper key handling.
    """
    
    stage_analysis = {
        'first_actions': Counter(),
        'last_actions': Counter(),
        'stage_positions': defaultdict(list),
        'stage_transitions': defaultdict(int)
    }
    
    for seq_data in action_sequences:
        sequence = seq_data['compressed_sequence']
        
        if len(sequence) > 0:
            # Track first and last actions
            stage_analysis['first_actions'][sequence[0]] += 1
            stage_analysis['last_actions'][sequence[-1]] += 1
            
            # Track position of each stage type
            for i, action in enumerate(sequence):
                relative_position = i / (len(sequence) - 1) if len(sequence) > 1 else 0
                stage_analysis['stage_positions'][action].append(relative_position)
            
            # Track transitions
            for i in range(len(sequence) - 1):
                transition = f"{sequence[i]} → {sequence[i+1]}"
                stage_analysis['stage_transitions'][transition] += 1
    
    # Calculate average positions and statistics - FIXED VERSION
    avg_positions = {}
    for stage, positions in stage_analysis['stage_positions'].items():
        avg_positions[stage] = {
            'mean_position': np.mean(positions),
            'std_position': np.std(positions),  # Fixed key name
            'early_occurrences': sum(1 for p in positions if p < 0.25) / len(positions),
            'mid_occurrences': sum(1 for p in positions if 0.25 <= p <= 0.75) / len(positions),
            'late_occurrences': sum(1 for p in positions if p > 0.75) / len(positions),
            'total_occurrences': len(positions)
        }
    
    return stage_analysis, avg_positions

# Fixed workflow stage analysis
stage_analysis, avg_positions = analyze_fixed_workflow_stages(action_sequences)

print("=== FIXED WORKFLOW STAGE ANALYSIS ===")

print("\n1. WORKFLOW INITIATION (First actions):")
for action, count in stage_analysis['first_actions'].most_common(8):
    pct = count / len(action_sequences) * 100
    print(f"   • {action.replace('_', ' ')}: {count} workflows ({pct:.1f}%)")

print("\n2. WORKFLOW COMPLETION (Final actions):")
for action, count in stage_analysis['last_actions'].most_common(8):
    pct = count / len(action_sequences) * 100
    print(f"   • {action.replace('_', ' ')}: {count} workflows ({pct:.1f}%)")

print("\n3. TEMPORAL POSITIONING (Average position in workflow):")
sorted_stages = sorted(avg_positions.items(), key=lambda x: x[1]['mean_position'])
for stage, stats in sorted_stages:
    print(f"   • {stage.replace('_', ' ')}: {stats['mean_position']:.3f} ± {stats['std_position']:.3f}")
    print(f"     Early: {stats['early_occurrences']:.1%}, Mid: {stats['mid_occurrences']:.1%}, Late: {stats['late_occurrences']:.1%}")

print("\n4. MOST FREQUENT TRANSITIONS:")
top_transitions = sorted(stage_analysis['stage_transitions'].items(), key=lambda x: x[1], reverse=True)[:12]
for i, (transition, count) in enumerate(top_transitions, 1):
    pct = count / sum(stage_analysis['stage_transitions'].values()) * 100
    print(f"   {i:2d}. {transition.replace('_', ' ')}: {count} times ({pct:.1f}%)")

## 4. Transition Matrix and Network Graph Analysis

Compute transition probabilities and create network graph visualization.

In [ ]:
def compute_fixed_transitions(action_sequences):
    """
    Compute transition matrix and probabilities from action sequences
    """
    # Get all unique actions
    all_actions = set()
    for seq_data in action_sequences:
        all_actions.update(seq_data['compressed_sequence'])
    action_list = sorted(list(all_actions))
    
    # Create action to index mapping
    action_to_idx = {action: i for i, action in enumerate(action_list)}
    
    # Initialize transition matrix
    n_actions = len(action_list)
    transition_counts = np.zeros((n_actions, n_actions))
    
    # Count transitions
    for seq_data in action_sequences:
        sequence = seq_data['compressed_sequence']
        for i in range(len(sequence) - 1):
            from_action = sequence[i]
            to_action = sequence[i + 1]
            from_idx = action_to_idx[from_action]
            to_idx = action_to_idx[to_action]
            transition_counts[from_idx, to_idx] += 1
    
    # Compute transition probabilities
    transition_probs = np.zeros((n_actions, n_actions))
    for i in range(n_actions):
        row_sum = transition_counts[i, :].sum()
        if row_sum > 0:
            transition_probs[i, :] = transition_counts[i, :] / row_sum
    
    return transition_counts, transition_probs, action_list, action_to_idx

def create_fixed_workflow_graph(transition_counts, action_list, avg_positions, action_to_idx, min_weight=30):
    """
    Create NetworkX graph from transition matrix
    """
    G = nx.DiGraph()
    
    # Add nodes with attributes
    for action in action_list:
        total_activity = transition_counts[action_to_idx[action], :].sum() + transition_counts[:, action_to_idx[action]].sum()
        G.add_node(action, total_activity=total_activity)
    
    # Add edges with weights above threshold
    for i, from_action in enumerate(action_list):
        for j, to_action in enumerate(action_list):
            if transition_counts[i, j] >= min_weight:
                G.add_edge(from_action, to_action, weight=transition_counts[i, j])
    
    return G

# Compute transitions and create graph
transition_counts, transition_probs, action_list, action_to_idx = compute_fixed_transitions(action_sequences)

print("=== TRANSITION MATRIX ANALYSIS ===")
print(f"Action categories: {len(action_list)}")
print(f"Categories: {[a.replace('_', ' ') for a in action_list]}")
print(f"Total transitions: {transition_counts.sum():,}")
print(f"Non-zero transitions: {np.count_nonzero(transition_counts)}")

# Create workflow graph
G = create_fixed_workflow_graph(transition_counts, action_list, avg_positions, action_to_idx, min_weight=100)

print(f"\n=== WORKFLOW GRAPH STATISTICS ===")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.3f}")
print(f"Strongly connected: {nx.is_strongly_connected(G)}")

# Analyze top transitions
transitions_list = []
for i, from_action in enumerate(action_list):
    for j, to_action in enumerate(action_list):
        if transition_counts[i, j] > 0:
            transitions_list.append({
                'from': from_action,
                'to': to_action,
                'count': transition_counts[i, j],
                'probability': transition_probs[i, j],
                'percentage': transition_counts[i, j] / transition_counts.sum() * 100
            })

transitions_analysis = pd.DataFrame(transitions_list).sort_values('count', ascending=False)

# print("\n=== TOP 12 TRANSITIONS ===")
# for i, (_, row) in enumerate(transitions_analysis.head(12).iterrows(), 1):
#     print(f"{i:2d}. {row['from'].replace('_', ' ')} → {row['to'].replace('_', ' ')}: {row['count']} ({row['percentage']:.1f}%)")

## 5. Network Visualization

Create the state transition graph visualization.

In [ ]:
# 1. Action Transition Network Graph
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Create evenly distributed circular layout with smaller radius to keep nodes inside
n_nodes = len(G.nodes())
if n_nodes > 0:
    angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False)
    pos = {}
    for i, node in enumerate(sorted(G.nodes())):
        x = np.cos(angles[i]) * 0.7
        y = np.sin(angles[i]) * 0.7
        pos[node] = (x, y)
    
# Calculate node sizes and radius for arrow positioning
activities = [G.nodes[node]['total_activity'] for node in G.nodes()]
min_activity = min(activities) if activities else 1
max_activity = max(activities) if activities else 1
    
node_sizes = []
node_radii = {}  # Store actual node radii for arrow positioning
for node in G.nodes():
    activity = G.nodes[node]['total_activity']
    if max_activity > min_activity:
        norm_activity = (activity - min_activity) / (max_activity - min_activity)
    else:
        norm_activity = 0.5
    size = 1500 + (norm_activity * 2500)  # Scale between 1500 and 4000
    node_sizes.append(size)
    # Calculate actual radius (matplotlib uses diameter in points, convert to data units)
    node_radii[node] = np.sqrt(size) * 0.003  # Approximate conversion

# Draw edges with improved arrow positioning
def calculate_edge_positions(pos, node_radii, from_node, to_node):
    """Calculate start and end positions for arrows to avoid overlapping nodes"""
    x1, y1 = pos[from_node]
    x2, y2 = pos[to_node]
    
    # Calculate direction vector
    dx = x2 - x1
    dy = y2 - y1
    distance = np.sqrt(dx**2 + dy**2)
    
    if distance > 0:
        # Normalize direction vector
        dx_norm = dx / distance
        dy_norm = dy / distance
        
        # Calculate start and end points outside the node circles
        start_radius = node_radii.get(from_node, 0.01)
        end_radius = node_radii.get(to_node, 0.01)
        
        start_x = x1 + dx_norm * start_radius
        start_y = y1 + dy_norm * start_radius
        end_x = x2 - dx_norm * end_radius  
        end_y = y2 - dy_norm * end_radius
        
        return (start_x, start_y), (end_x, end_y)
    else:
        return (x1, y1), (x2, y2)

# Draw edges with variable width but uniform arrow size
weights = [G[u][v]['weight'] for u, v in G.edges()]
max_weight = max(weights) if weights else 1

for (u, v) in G.edges():
    weight = G[u][v]['weight']
    edge_width = (weight / max_weight) * 10 + 1  # Variable edge width
    edge_alpha = 0.5 + (weight / max_weight) * 0.4  # Variable transparency
    
    # Calculate proper edge positions
    start_pos, end_pos = calculate_edge_positions(pos, node_radii, u, v)
    
    # Variable width but uniform arrow size using shrinkA/shrinkB
    ax.annotate('', xy=end_pos, xytext=start_pos, 
                arrowprops=dict(arrowstyle='->', lw=edge_width, alpha=edge_alpha, 
                                color='gray', connectionstyle="arc3,rad=0.1",
                                shrinkA=0, shrinkB=0,  # Control arrow positioning
                                mutation_scale=15))    # Fixed arrow size
    
# Draw nodes
node_colors = [refined_category_colors.get(node, '#CCCCCC') for node in G.nodes()]
    
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, 
                       node_size=node_sizes, alpha=0.9, edgecolors='black', linewidths=2)
    
# Add labels
labels = {node: node.replace('_', '\n') for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=11, font_weight='bold', font_color='black')

ax.set_title('Action Transition Network', fontsize=16, fontweight='bold', pad=20)
ax.axis('off')

# Set axis limits to ensure all nodes are visible
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Action Transition Network Graph
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
node_sizes = []
node_radii = {}  # Store actual node radii for arrow positioning
for node in G.nodes():
    activity = G.nodes[node]['total_activity']
    if max_activity > min_activity:
        norm_activity = (activity - min_activity) / (max_activity - min_activity)
    else:
        norm_activity = 0.5
    size = 1500 + (norm_activity * 2500)  # Scale between 1500 and 4000
    node_sizes.append(size)
    # Calculate actual radius (matplotlib uses diameter in points, convert to data units)
    node_radii[node] = np.sqrt(size) * 0.003  # Approximate conversion

# Draw edges with variable width but uniform arrow size
weights = [G[u][v]['weight'] for u, v in G.edges()]
max_weight = max(weights) if weights else 1

for (u, v) in G.edges():
    weight = G[u][v]['weight']
    edge_width = (weight / max_weight) * 10 + 1  # Variable edge width
    edge_alpha = 0.5 + (weight / max_weight) * 0.4  # Variable transparency
    
    # Calculate proper edge positions
    start_pos, end_pos = calculate_edge_positions(pos, node_radii, u, v)
    
    # Draw edges as lines without arrows
    ax.plot([start_pos[0], end_pos[0]], [start_pos[1], end_pos[1]], 
            color='gray', linewidth=edge_width, alpha=edge_alpha)
    
# Draw nodes
node_colors = [refined_category_colors.get(node, '#CCCCCC') for node in G.nodes()]
    
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, 
                       node_size=node_sizes, alpha=0.6, edgecolors='black', linewidths=2)
    
# Add labels
labels = {node: node.replace('_', '\n') for node in G.nodes()}
# nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=11, font_weight='bold', font_color='black')
ax.axis('off')

# Set axis limits to ensure all nodes are visible
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)

plt.tight_layout()
plt.savefig("action transition.pdf", dpi=300)
plt.show()

In [ ]:
# 2. Transition Probability Matrix (Separate Figure)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

clean_labels = [label.replace('_', ' ')[:15] for label in action_list]
mask = transition_probs == 0

sns.heatmap(transition_probs, annot=True, fmt='.2f', cmap='YlOrRd', 
            xticklabels=clean_labels, yticklabels=clean_labels, 
            ax=ax, mask=mask, square=True, cbar_kws={'shrink': 0.8})

ax.set_title('Action Transition Probabilities Matrix', fontweight='bold', fontsize=16, pad=20)
ax.set_xlabel('To Action', fontsize=12, fontweight='bold')
ax.set_ylabel('From Action', fontsize=12, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

## Summary: Action Transition Patterns with Refined Categories

### Key Findings

Using the refined 10-category system, the action transition analysis reveals:

**Dominant Workflow Patterns:**
1. **Data Exploration ↔ Study Design Analysis** - Core scientific reasoning loop (15.6% of transitions)
2. **Data Exploration ↔ Multi-source Integration** - Data combination workflow (13.4% of transitions)  
3. **Data Exploration ↔ Statistical Analysis** - Analysis progression (10.1% of transitions)

**Temporal Workflow Progression:**
- **Early Phase**: Data Exploration and Study Design Analysis dominate workflow initiation
- **Mid Phase**: Multi-source Integration becomes critical for data combination
- **Late Phase**: Statistical Analysis and Effect Size Calculation increase, Results Export peaks

**Scientific Workflow Insights:**
- **Data Exploration serves as the central hub** - agents constantly return to exploring data
- **Study Design Analysis is critical early** - understanding experimental setup is essential
- **Clear progression pattern** from exploration → design understanding → integration → statistical testing
- **Bi-directional transitions** show iterative refinement and validation

This transition analysis confirms that AI agents follow sophisticated scientific reasoning patterns, with **Data Exploration acting as the central coordinating activity** that connects all other research tasks.